In [15]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [16]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/NGC-3217_2_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/NGC-3217_2_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     359   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        40   (2094, 964)   float32   


In [17]:
image_cut = image[67:167, 0:1890]
image_error_cut = image_error[67:167, 0:1890]

In [18]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.03191
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [19]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4)# +
        #sersic_1d(x_hr, params5)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [20]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/PSF/Real_seeing_NGC3217_2.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [21]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, a, b):
    func = model_convolved(x, params1, params2, params3, params4)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    #choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area#+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area]#, (choc_area*100)/sum_area ]    

## Halpha

In [22]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))

sky_err = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Propag_errori/Sky_errors.txt")
real_errors_HA = np.sqrt(radial_profile_error_HA**2+(sky_err[2])**2+(0.04*radial_profile_HA)**2)

fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Fit/Halpha_fit.csv", index_col=0)

In [23]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

#data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
#                      fit_HA['Component 5'].iloc[3] -1, fit_HA['Component 5'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

2.334077726535682
2.193956412540359
6.6510461812328785
6.427246308856715
2.9007310365516648
2.904262326662208
0.6754759567011678
0.699947715981382
              Peak 1     Peak 2     Peak 3        Peak 4
Blue    8.256629e+01   0.051325   0.010889  3.092052e-04
Green   1.626200e+01  99.043223  49.972591  2.484939e+01
Purple  1.069826e-17   0.118243  47.331567  4.515173e-08
Orange  1.171702e+00   0.787210   2.684952  7.515030e+01


In [24]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

#data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
#                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)


df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

4.772648445314125
4.434919201593002
13.626109240651669
13.073316008233707
5.530630435609846
5.540966680567671
1.3702891295348891
1.407531758165572
              Peak 1     Peak 2     Peak 3        Peak 4
Blue    8.405034e+01   0.041076   0.009501  3.632041e-04
Green   1.482771e+01  98.900878  47.243474  2.604863e+01
Purple  3.619275e-18   0.266078  49.820953  3.389607e-07
Orange  1.121949e+00   0.791968   2.926072  7.395101e+01


In [25]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'],
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

#data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
#                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

7.9242011954790055
7.5332397368302395
23.275070025996904
22.65504442211542
10.511593292393979
10.529895613908634
2.518239748496641
2.571399091714993
              Peak 1     Peak 2     Peak 3     Peak 4
Blue    8.056402e+01   0.052387   0.010751   0.000419
Green   1.808274e+01  98.410105  51.258936  28.601640
Purple  1.414917e-16   0.607599  45.636423   0.000003
Orange  1.353244e+00   0.929910   3.093890  71.397937


## HBeta

In [26]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
real_errors_HB = np.sqrt(radial_profile_error_HB**2+(sky_err[2])**2+(0.04*radial_profile_HB)**2)
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Fit/Hbeta_fit.csv", index_col = 0)

In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)


df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6083808576985144
0.6241706222777725
2.6003431661466863
2.682314667798959
1.2133366591353023
1.2143580459598264
0.3673075597069683
0.36304914117537546
              Peak 1     Peak 2     Peak 3     Peak 4
Blue    9.136432e+01   4.007103   5.380078   7.825961
Green   6.464275e+00  93.854199  40.119651   3.216391
Purple  2.332474e-08   0.620047  48.776532   0.006356
Orange  2.171405e+00   1.518651   5.723739  88.951293


In [28]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.091263541201279
1.1036885255242341
5.685614693450138
5.951096093373078
2.3815066546520622
2.385037218570714
0.7510948835091574
0.7413724116147238
              Peak 1     Peak 2     Peak 3     Peak 4
Blue    9.127934e+01   3.510982   5.297052   7.851205
Green   6.393029e+00  94.126053  34.721170   3.527945
Purple  1.215583e-08   0.909032  53.849921   0.013305
Orange  2.327633e+00   1.453933   6.131858  88.607545


In [29]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


2.142486857790629
2.1919309385505037
10.051299120838197
10.412686354542275
4.672162671904305
4.678663677489528
1.4158209608717596
1.4048910977949873
              Peak 1     Peak 2     Peak 3     Peak 4
Blue    9.089520e+01   3.994701   5.418577   8.344710
Green   6.725525e+00  92.987967  37.797502   3.846202
Purple  4.134673e-08   1.364719  50.495704   0.027438
Orange  2.379273e+00   1.652613   6.288217  87.781649


## NII

In [30]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Fit/NII_fit.csv", index_col = 0)

In [31]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.9162546839991674
0.9844977528083174
3.786912609927641
3.858094012814167
1.4233920778151745
1.424234962860729
0.37226850865315175
0.3795986281773859
              Peak 1     Peak 2     Peak 3     Peak 4
Blue    6.714101e+01   0.186250   0.131692   0.040971
Green   3.106872e+01  98.842148  60.409690  39.189927
Purple  6.145621e-15   0.114121  36.005520   0.000001
Orange  1.790265e+00   0.857481   3.453097  60.769101


In [32]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],  
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],  
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.7666111884833773
1.8780056001223207
7.65870485170225
7.734308715769384
2.7280550890634125
2.7309214532261614
0.8024870156227856
0.8218794657362763
              Peak 1     Peak 2     Peak 3     Peak 4
Blue    6.774718e+01   0.163869   0.124408   0.041243
Green   3.043706e+01  98.726583  57.539923  38.137032
Purple  2.301714e-15   0.234458  38.585861   0.000006
Orange  1.815754e+00   0.875090   3.749808  61.821720


In [33]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

3.152087715040969
3.3044517496708483
13.150651069535103
13.286105879356006
5.286352420225139
5.292841969210459
1.54480039531208
1.5681227201222812
              Peak 1     Peak 2     Peak 3     Peak 4
Blue    6.353825e+01   0.197050   0.131325   0.043455
Green   3.442355e+01  98.304146  60.947409  39.857673
Purple  3.977798e-14   0.476464  35.031095   0.000032
Orange  2.038205e+00   1.022341   3.890170  60.098840


## SII

In [34]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Fit/SII_fit.csv", index_col = 0)

In [35]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)


df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.7148435020874662
0.7152350497731212
2.3339073239958545
2.416208438872414
0.9515079934563143
0.9405754106061116
0.29870247470358324
0.3021271826829473
           Peak 1        Peak 2        Peak 3         Peak 4
Blue    58.410107  3.628267e-17  8.188508e-42  3.416231e-149
Green   41.475916  9.980487e+01  7.651687e+01   5.558410e+01
Purple   0.000000  0.000000e+00  2.229019e+01   0.000000e+00
Orange   0.113977  1.951307e-01  1.192949e+00   4.441590e+01


In [36]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.3349330138837854
1.3333866427433314
4.649039769104201
4.784415037489114
1.8344514474090037
1.820642506095431
0.6309122219211809
0.6402281772160205
           Peak 1        Peak 2        Peak 3         Peak 4
Blue    57.562029  1.216109e-17  2.831555e-42  1.158376e-139
Green   42.323094  9.978918e+01  7.368006e+01   5.480593e+01
Purple   0.000000  0.000000e+00  2.497572e+01   0.000000e+00
Orange   0.114877  2.108239e-01  1.344220e+00   4.519407e+01


In [37]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],  
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],  
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

2.4301996378822057
2.4280863821127063
8.293252904389423
8.48049444414357
3.5872218785882817
3.5436194860052392
1.2179739857843073
1.2301864683147432
           Peak 1        Peak 2        Peak 3         Peak 4
Blue    53.068261  6.304208e-15  1.606378e-37  2.814474e-130
Green   46.804513  9.975999e+01  7.672594e+01   5.702809e+01
Purple   0.000000  8.587124e-04  2.187789e+01   0.000000e+00
Orange   0.127226  2.391546e-01  1.396169e+00   4.297191e+01


## OIII

In [38]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_2/Fit/OIII_fit.csv", index_col = 0)

In [39]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)


df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5881817708599025
0.5894272718546673
2.2826071461363107
2.296529610406907
0.919362694633228
0.9169390739584959
0.37662112353384364
0.3783062903709409
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    93.206451   6.028657   9.580139  10.142047
Green    4.224661  90.547736  39.173456   1.525824
Purple   0.009719   1.202500  41.266955   0.828507
Orange   2.559169   2.221107   9.979450  87.503622


In [40]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],  
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],  
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.088055794410844
1.08607371796788
4.869106102260372
4.947196213654572
1.8575790660297469
1.8505777148936464
0.7845648580115336
0.7885781966485174
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    93.361378   5.430276   9.168542  10.001494
Green    3.985068  90.953456  32.645586   1.673165
Purple   0.008319   1.429534  47.788008   1.016336
Orange   2.645234   2.186735  10.397864  87.309005


In [41]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

2.1282923551155783
2.134343835298286
8.525958357754105
8.645293726954593
3.7066663402283146
3.6993594487644286
1.525399489202095
1.5300125276736227
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    93.030017   6.231224   9.217932  10.306771
Green    4.244227  89.456625  35.084669   1.783038
Purple   0.009531   1.801920  45.216245   1.162014
Orange   2.716226   2.510232  10.481154  86.748177
